In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os

# --- Config ---
DATA_PATH = "WISDM_ar_v1.1_raw.txt" # Update if needed
EXPORT_NAME = "neuron_config"
WINDOW_SIZE = 80
STEP_SIZE = 40

# --- 1. DATA LOADER ---
def load_data(path):
    print(f"Loading {path}...")
    data = []
    with open(path, 'r') as f:
        for line in f:
            try:
                line = line.strip().replace(';', '')
                parts = line.split(',')
                if len(parts) == 6:
                    # user, activity, timestamp, x, y, z
                    data.append([int(parts[0]), parts[1], float(parts[3]), float(parts[4]), float(parts[5])])
            except: continue
    return pd.DataFrame(data, columns=['user', 'activity', 'x', 'y', 'z'])

# --- 2. PYTHON FEATURE EXTRACTION (Exact match to your C code) ---
def get_features(segment):
    xs = segment['x'].values
    ys = segment['y'].values
    zs = segment['z'].values
    
    # 1. Means
    mx, my, mz = np.mean(xs), np.mean(ys), np.mean(zs)
    
    # 2. Pos Counts (C code: value > 0)
    px = np.sum(xs > 0)
    py = np.sum(ys > 0)
    pz = np.sum(zs > 0)
    
    # 3. Std Dev (C code uses population std: div by N)
    sx = np.std(xs) 
    sy = np.std(ys)
    sz = np.std(zs)
    
    # 4. SMA
    sma = (np.sum(np.abs(xs)) + np.sum(np.abs(ys)) + np.sum(np.abs(zs))) / len(xs)
    
    # Order MUST match the mapping in main.c
    # [MeanX, MeanY, MeanZ, PosX, PosY, PosZ, StdX, StdY, StdZ, SMA]
    return np.array([mx, my, mz, px, py, pz, sx, sy, sz, sma], dtype=np.float32)

def prepare_dataset(df):
    X, y = [], []
    for i in range(0, len(df) - WINDOW_SIZE, STEP_SIZE):
        seg = df.iloc[i : i+WINDOW_SIZE]
        label = 1.0 if seg['activity'].iloc[0] == 'Walking' else 0.0
        X.append(get_features(seg))
        y.append(label)
    return np.array(X), np.array(y)

# --- 3. EXPORT TO C ---
def export_c(model, filename):
    w, b = model.layers[0].get_weights()
    w_flat = w.flatten()
    b_val = b[0]
    
    print(f"Exporting {len(w_flat)} weights to {filename}.c...")
    
    with open(f"{filename}.h", "w") as f:
        f.write(f"#ifndef {filename.upper()}_H\n#define {filename.upper()}_H\n\n")
        f.write("#define NUM_FEATURES 10\n")
        f.write("float neuron_predict(float *features);\n")
        f.write("#endif\n")
        
    with open(f"{filename}.c", "w") as f:
        f.write(f'#include "{filename}.h"\n#include <math.h>\n\n')
        f.write(f"static const float W[10] = {{\n    ")
        for i, val in enumerate(w_flat):
            f.write(f"{val:.8f}f, ")
            if (i+1)%5==0: f.write("\n    ")
        f.write(f"\n}};\n\n")
        f.write(f"static const float B = {b_val:.8f}f;\n\n")
        f.write("float neuron_predict(float *x) {\n")
        f.write("    float z = B;\n")
        f.write("    for(int i=0; i<10; i++) z += x[i] * W[i];\n")
        f.write("    return 1.0f / (1.0f + expf(-z));\n")
        f.write("}\n")

# --- MAIN ---
if __name__ == "__main__":
    df = load_data(DATA_PATH)
    
    # Split
    df_train = df[df['user'] <= 28]
    
    print("Extracting features (Training)...")
    X_train, y_train = prepare_dataset(df_train)
    
    # Simple Normalization (Optional but recommended for Neural Networks)
    # Ideally you export mean/std like SVM, but single neuron often adapts.
    
    print("Training Keras Model...")
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(10,)),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=1)
    
    export_c(model, EXPORT_NAME)
    print("DONE.")

Loading WISDM_ar_v1.1_raw.txt...
Extracting features (Training)...
Training Keras Model...
Epoch 1/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.3160 - loss: 4.5495
Epoch 2/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 927us/step - accuracy: 0.5995 - loss: 0.8292
Epoch 3/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 928us/step - accuracy: 0.6960 - loss: 0.5920
Epoch 4/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7424 - loss: 0.5245
Epoch 5/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7631 - loss: 0.4955
Epoch 6/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7742 - loss: 0.4833
Epoch 7/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7789 - loss: 0.4765
Epoch 8/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 957us/step - accuracy: 0.7791 - loss: 0.4715
Epoch 9/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 914us/step - accuracy: 0.7805 - loss: 0.4677
Epoch 10/20
643/643 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7796 - loss: 0.4649
Epoch 11/20
643/643 ━━━━━━━━━━━━━━━━━━━━